In [ ]:
import numpy as np
import xarray as xr
import pint_xarray
import matplotlib.pylab as plt
import geopandas as gpd

from pism_terra.grids import sample_points


In [ ]:

def interp_to_height(field, height, target, dim="pressure_level"):
    """
    Interpolate a pressure-level field to heights, linearly in z.

    Each target height gets the linear blend of the two levels that bracket it.
    Heights below the lowest or above the highest level come back NaN.

    field, height : DataArrays on (..., dim, ...); height in m, rising along dim.
    target : heights in m. A DataArray with a "z" dim gives a 1-D height axis;
             a (y, x) DataArray such as the orography gives values on that surface.
    """
    result = None
    for k in range(field.sizes[dim] - 1):
        z0, z1 = height.isel({dim: k}, drop=True), height.isel({dim: k + 1}, drop=True)
        f0, f1 = field.isel({dim: k}, drop=True), field.isel({dim: k + 1}, drop=True)
        w = (target - z0) / (z1 - z0)
        value = (f0 + w * (f1 - f0)).where((w >= 0) & (w <= 1))
        result = value if result is None else result.fillna(value)
    result.attrs = dict(field.attrs)
    return result

In [ ]:
ds = xr.open_dataset("s3://pism-cloud-data/glacier/input/s4f/climate/carra2_daily_RGI2000-v7.0-C-01-04374.zarr", 
                     engine="zarr",
                     storage_options={"anon": True})

z = xr.DataArray(np.arange(0.0, 5501.0, 100.0), dims="z", attrs={"units": "m", "positive": "up"})
t_z = interp_to_height(ds["air_temp"], ds["geopotential_height"], z).assign_coords(z=z)
t_z = t_z.where(t_z["z"] >= ds["orography"]).transpose("time", "z", "y", "x")

monthly = ds["air_temp"].resample(time="1MS").mean()
jja = monthly.sel(time=monthly["time"].dt.month.isin([6, 7, 8]))
jja = xr.merge([jja, ds.spatial_ref])



In [ ]:

# monthly = carra2["air_temp"].resample(time="1MS").mean()
jja = t_z.sel(time=t_z["time"].dt.month.isin([6, 7, 8]))
jja.name = "air_temp"
jja = xr.merge([jja.to_dataset(), ds.drop_vars(["air_temp", "geopotential_height"])], join="outer", compat="no_conflicts")


outlines = gpd.read_file("/home/andy/storstrommen/terra/glacier_s4f_input/RGI2000-v7.0-C-01-04374/input/rgi_RGI2000-v7.0-C-01-04374-G.gpkg")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
from matplotlib.dates import DateFormatter, date2num

t_all = carra2_sites["air_temp"] - 273.15       # (time, z, site), °C, NaN below CARRA2 terrain
z = carra2_sites["z"].values                    # (z,), m
sites = carra2_sites["site"].values
times = pd.DatetimeIndex(carra2_sites["time"].values)
colors = date2num(times)
norm = dict(cmap="viridis", vmin=colors.min(), vmax=colors.max())



In [ ]:
import numpy as np
import xarray as xr


def fit_to_ground(temp, height, ground, below=1, dim="pressure_level"):
    """
    Fit T = a + b z through the pressure levels each day and evaluate it at the ground.

    temp, height : (..., dim) temperatures and level heights in m, levels ordered bottom up.
    ground       : ground height in m, broadcastable (e.g. orography on site).
    below        : levels below the ground to include besides every level above it;
                   0 = above only, None = all levels.
    Returns t_ground (K), lapse_rate (K/km) and n_levels.
    """
    if below is None:
        use = xr.ones_like(height, dtype=bool)
    else:
        # Levels run bottom up, so the ones below the ground come first.
        n_below = (height < ground).sum(dim)
        index = xr.DataArray(np.arange(height.sizes[dim]), dims=dim)
        use = index >= n_below - below
    z, t = height.where(use), temp.where(use)
    n = use.sum(dim)
    z_mean, t_mean = z.mean(dim), t.mean(dim)
    slope = ((z - z_mean) * (t - t_mean)).sum(dim) / ((z - z_mean) ** 2).sum(dim)
    slope = slope.where(n >= 2)
    return xr.Dataset({
        "t_ground": (t_mean + slope * (ground - z_mean)).assign_attrs(units="K"),
        "lapse_rate": (1000 * slope).assign_attrs(units="K km-1"),
        "n_levels": n,
    })

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fit = fit_to_ground(pts["air_temp"], pts["geopotential_height"], pts["orography"], below=1)
bias = (fit["t_ground"] - pts["air_temp_2m"]).rename("bias")

df = bias.to_dataframe().reset_index()
df["month"] = df["time"].dt.month
months = list(range(1, 13))
data = [df.loc[df["month"] == m, "bias"].dropna().to_numpy() for m in months]

fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
parts = ax.violinplot(data, positions=months, widths=0.85, showmedians=True, showextrema=False)
for body in parts["bodies"]:
    body.set_facecolor("0.8")
    body.set_edgecolor("0.4")
    body.set_alpha(0.8)
parts["cmedians"].set_color("k")

monthly = df.groupby(["month", "site"], sort=False)["bias"].median().unstack()
offsets = np.linspace(-0.25, 0.25, bias.sizes["site"])
for color, dx, site in zip(plt.cm.tab10.colors, offsets, bias["site"].values):
    ax.plot(np.array(months) + dx, monthly.loc[months, site], "o", ms=5, color=color,
            mec="k", mew=0.4, label=str(site), zorder=3)

ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(months, pd.to_datetime([f"2005-{m:02d}-01" for m in months]).strftime("%b"))
ax.set_ylabel("Level fit at ground − 2 m air temperature (K)")
ax.legend(title="site median", ncol=3, fontsize=8, loc="upper left")
ax.grid(axis="y", alpha=0.3)
fig.savefig("carra2_lapse_bias.png", dpi=300)

In [ ]:
sites = list(bias["site"].values)
labels = pd.to_datetime([f"2005-{m:02d}-01" for m in months]).strftime("%b").str[0]   # J F M ...

fig, axes = plt.subplots(3, 2, figsize=(11, 10), sharex=True, sharey=True, constrained_layout=True)
axes = axes.ravel()
for ax, site in zip(axes, sites):
    d = df[df["site"] == site]
    data = [d.loc[d["month"] == m, "bias"].dropna().to_numpy() for m in months]
    keep = [i for i, x in enumerate(data) if x.size > 1]          # a violin needs at least 2 values
    parts = ax.violinplot([data[i] for i in keep], positions=[months[i] for i in keep],
                          widths=0.85, showmedians=True, showextrema=False)
    for body in parts["bodies"]:
        body.set_facecolor("0.8")
        body.set_edgecolor("0.4")
        body.set_alpha(0.8)
    parts["cmedians"].set_color("k")

    orog = float(pts["orography"].sel(site=site))
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(f"{site}  (CARRA2 terrain {orog:.0f} m, annual median {d['bias'].median():+.1f} K)", fontsize=10)
    ax.grid(axis="y", alpha=0.3)

for ax in axes[len(sites):]:
    ax.set_visible(False)
for ax in axes[-2:]:
    ax.set_xticks(months, labels)
for ax in axes[::2]:
    ax.set_ylabel("Fit at ground − 2 m T (K)")
fig.suptitle("Daily lapse-rate extrapolation vs CARRA2 2 m temperature, per site")
fig.savefig("carra2_lapse_bias_sites.png", dpi=300)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

crs = ds["spatial_ref"].attrs["crs_wkt"]

# 1. Daily fit through the levels at every cell, evaluated at CARRA2's terrain.
fit = fit_to_ground(ds["air_temp"], ds["geopotential_height"], ds["orography"], below=1)
bias = (fit["t_ground"] - ds["air_temp_2m"]).load()                 # (time, y, x), K

# 2. On or off glacier: is the cell centre inside an outline?
xx, yy = np.meshgrid(ds["x"].values, ds["y"].values)
centres = gpd.GeoDataFrame(geometry=gpd.points_from_xy(xx.ravel(), yy.ravel()), crs=crs)
inside = centres.sjoin(outlines.to_crs(crs)[["geometry"]], predicate="within", how="left")
on_glacier = ~inside.groupby(level=0)["index_right"].first().isna()
category = xr.DataArray(
    np.where(on_glacier.to_numpy().reshape(xx.shape), "on_glacier", "off_glacier"), dims=("y", "x")
)

# 3. 500 m bins of the CARRA2 terrain elevation.
width = 500.0
orog = ds["orography"].load()
edges = np.arange(np.floor(float(orog.min()) / width) * width, float(orog.max()) + width, width)
centre = 0.5 * (edges[:-1] + edges[1:])
elev_bin = xr.DataArray(np.digitize(orog.values, edges) - 1, dims=("y", "x"))

df = pd.DataFrame({
    "bias": bias.values.reshape(bias.sizes["time"], -1).ravel(),
    "bin": np.tile(elev_bin.values.ravel(), bias.sizes["time"]),
    "category": np.tile(category.values.ravel(), bias.sizes["time"]),
    "elevation": np.tile(orog.values.ravel(), bias.sizes["time"]),
}).dropna()
df["centre"] = centre[df["bin"].to_numpy()]

# 4. Violins: elevation up the y axis, one pair per bin.
styles = {"off_glacier": ("tab:brown", -0.25), "on_glacier": ("tab:blue", +0.25)}
fig, ax = plt.subplots(figsize=(8, 10), constrained_layout=True)
for cat, (color, shift) in styles.items():
    d = df[df["category"] == cat]
    present = [b for b in range(len(centre)) if (d["bin"] == b).sum() > 1]
    if not present:
        continue
    parts = ax.violinplot([d.loc[d["bin"] == b, "bias"].to_numpy() for b in present],
                          positions=centre[present] + shift * width, widths=0.45 * width,
                          orientation="horizontal", showmedians=True, showextrema=False)
    for body in parts["bodies"]:
        body.set_facecolor(color)
        body.set_edgecolor("k")
        body.set_linewidth(0.5)
        body.set_alpha(0.7)
    parts["cmedians"].set_color("k")
    ax.plot([], [], "s", color=color, alpha=0.7, label=cat.replace("_", " "))

    # Number of grid cells in each bin, at the right edge.
    cells = np.bincount(elev_bin.values[category.values == cat], minlength=len(centre))
    for b in present:
        ax.annotate(str(cells[b]), (1.0, centre[b] + shift * width), xycoords=("axes fraction", "data"),
                    xytext=(3, 0), textcoords="offset points", va="center", fontsize=7, color=color)

def fit_piecewise(z, y, n_candidates=400, min_points=3):
    """
    Fit a continuous two-segment line y(z) with a scanned breakpoint.

    y = a + k_below * min(z - z_b, 0) + k_above * max(z - z_b, 0). For each
    candidate z_b the rest is linear least squares; the z_b with the smallest
    squared error wins. Each segment keeps at least min_points.
    """
    z, y = np.asarray(z, float), np.asarray(y, float)
    keep = np.isfinite(z) & np.isfinite(y)
    z, y = z[keep], y[keep]
    if z.size < 2 * min_points:
        return None
    zs = np.sort(z)
    best = None
    for zb in np.linspace(zs[min_points - 1], zs[-min_points], n_candidates):
        dz = z - zb
        if (dz < 0).sum() < min_points - 1 or (dz > 0).sum() < min_points - 1:
            continue
        design = np.column_stack([np.ones_like(z), np.minimum(dz, 0), np.maximum(dz, 0)])
        coef, *_ = np.linalg.lstsq(design, y, rcond=None)
        sse = float(((y - design @ coef) ** 2).sum())
        if best is None or sse < best[0]:
            best = (sse, zb, coef)
    sse, zb, (a, k_below, k_above) = best
    return {"z_break": zb, "y_break": a, "k_below": k_below, "k_above": k_above,
            "rmse": np.sqrt(sse / z.size), "n": z.size}


def piecewise_line(fit, z):
    dz = np.asarray(z, float) - fit["z_break"]
    return fit["y_break"] + fit["k_below"] * np.minimum(dz, 0) + fit["k_above"] * np.maximum(dz, 0)


# Median bias per bin and category, and how many grid cells each rests on.
medians = df.groupby(["category", "centre"])["bias"].agg(["median", "count"]).reset_index()
medians["cells"] = medians["count"] // bias.sizes["time"]
min_cells = 3                                   # bins with fewer cells stay out of the fit

fits = {}
for cat, (color, shift) in styles.items():
    m = medians[medians["category"] == cat]
    used = m[m["cells"] >= min_cells]
    fit = fit_piecewise(used["centre"], used["median"])
    if fit is None:
        continue
    fits[cat] = fit
    zline = np.linspace(used["centre"].min(), used["centre"].max(), 200)
    ax.plot(piecewise_line(fit, zline), zline, color=color, lw=2.2, zorder=4)
    ax.plot(fit["y_break"], fit["z_break"], "D", ms=7, color=color, mec="k", zorder=6)

fits = pd.DataFrame(fits).T
fits["k_below_per_km"] = 1000 * fits["k_below"]
fits["k_above_per_km"] = 1000 * fits["k_above"]
print(fits[["z_break", "y_break", "k_below_per_km", "k_above_per_km", "rmse", "n"]].round(2))
ax.axvline(0, color="k", lw=0.8)
ax.set_yticks(edges)
ax.set_ylim(edges[0], edges[-1])
ax.set_xlabel("Level fit at ground − 2 m air temperature (K)")
ax.set_ylabel("CARRA2 terrain elevation (m), 250 m bins")
ax.set_title("Daily lapse-rate extrapolation vs CARRA2 2 m temperature, every grid cell")
ax.legend(loc="upper right", title="cell centre")
ax.grid(axis="x", alpha=0.3)
fig.savefig("carra2_bias_by_elevation.png", dpi=300)

In [ ]:
df.groupby(["centre", "category"])["bias"].median().unstack("category")